In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E7.2 · Bronze → Silver
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Transformar los datos Bronze a estructuras Silver
# MAGIC conformadas y tipadas.
# MAGIC
# MAGIC ### Principios
# MAGIC
# MAGIC - Bronze conserva el dato original.
# MAGIC - Silver interpreta, tipa, normaliza y conforma.
# MAGIC - Los registros inválidos de Review no se eliminan:
# MAGIC   se envían a cuarentena.
# MAGIC - Los posibles duplicados lógicos de Review se conservan
# MAGIC   como señal para Trust & Safety.
# MAGIC - Las reglas Trust & Safety NO se ejecutan en este notebook.
# MAGIC - El DQ Silver se ejecuta posteriormente como una Task separada.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import uuid
from datetime import datetime, timezone


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"
SILVER_SCHEMA = "yelp_silver"

BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"

PIPELINE_NAME = "E7_02_bronze_to_silver"

silver_execution_id = str(uuid.uuid4())

execution_started_at = (
    datetime
    .now(timezone.utc)
    .isoformat()
)


print("=" * 70)
print("YELPCONNECT - BRONZE -> SILVER")
print("=" * 70)

print(f"Pipeline      : {PIPELINE_NAME}")
print(f"Execution ID  : {silver_execution_id}")
print(f"Inicio UTC    : {execution_started_at}")

print("=" * 70)


# COMMAND ----------

# ============================================================
# 3. CREAR / VALIDAR SCHEMA SILVER
# ============================================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{SILVER_SCHEMA}
""")


print(
    f"✔ Schema disponible: "
    f"{CATALOG}.{SILVER_SCHEMA}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 4. BUSINESS
# MAGIC
# MAGIC Transformaciones:
# MAGIC
# MAGIC - Normalización de ciudad y estado.
# MAGIC - Tipado numérico.
# MAGIC - `is_open` → boolean.
# MAGIC - `categories` → array.
# MAGIC - Attributes y Hours se conservan como estructuras complejas.
# MAGIC - Deduplicación por `business_id`.


# COMMAND ----------

# ============================================================
# 4. SILVER BUSINESS
# ============================================================

brz_business = (
    spark.table(
        f"{BRONZE}.brz_business"
    )
)


slv_business = (
    brz_business

    .select(

        F.col("business_id"),

        F.trim(
            F.col("name")
        ).alias("nombre"),

        F.trim(
            F.col("address")
        ).alias("direccion"),

        F.initcap(
            F.trim(
                F.col("city")
            )
        ).alias("ciudad"),

        F.upper(
            F.trim(
                F.col("state")
            )
        ).alias("estado"),

        F.trim(
            F.col("postal_code")
        ).alias("codigo_postal"),

        F.col("latitude")
            .cast("double")
            .alias("latitud"),

        F.col("longitude")
            .cast("double")
            .alias("longitud"),

        F.col("stars")
            .cast("double")
            .alias("estrellas"),

        F.col("review_count")
            .cast("long")
            .alias("n_resenas"),

        (
            F.col("is_open")
            .cast("int") == 1
        ).alias("esta_abierto"),

        # ARRAY de categorías
        F.when(
            F.col("categories").isNull(),
            F.lit(None)
        )
        .otherwise(
            F.split(
                F.col("categories"),
                r",\s*"
            )
        )
        .alias("categorias"),

        # Conservamos estructuras complejas
        F.col("attributes")
            .alias("atributos"),

        F.col("hours")
            .alias("horarios"),

        # Linaje desde Bronze
        F.col("_archivo_origen"),

        F.col("_execution_id")
            .alias("_bronze_execution_id"),

        F.col("_record_checksum")
            .alias("_bronze_record_checksum"),

        F.lit(
            silver_execution_id
        ).alias("_silver_execution_id"),

        F.current_timestamp()
            .alias("_silver_processed_at"),

        F.lit("silver")
            .alias("_layer")
    )

    .filter(
        F.col("business_id")
        .isNotNull()
    )

    .dropDuplicates(
        ["business_id"]
    )
)


(
    slv_business

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_business"
    )
)


business_count = (
    spark.table(
        f"{SILVER}.slv_business"
    )
    .count()
)


print(
    f"✔ slv_business: "
    f"{business_count:,} filas"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. USER
# MAGIC
# MAGIC Transformaciones:
# MAGIC
# MAGIC - `yelping_since` → TIMESTAMP.
# MAGIC - Contadores → tipos numéricos.
# MAGIC - `elite` → ARRAY.
# MAGIC - `friends` → ARRAY.
# MAGIC - Deduplicación por `user_id`.


# COMMAND ----------

# ============================================================
# 5. SILVER USER
# ============================================================

brz_user = (
    spark.table(
        f"{BRONZE}.brz_user"
    )
)


# Función auxiliar STRING CSV → ARRAY
def csv_to_array(column_name):

    return (
        F.when(
            F.col(column_name).isNull()
            |
            (
                F.trim(
                    F.col(column_name)
                ) == ""
            ),

            F.expr(
                "array()"
            ).cast(
                "array<string>"
            )
        )

        .otherwise(
            F.split(
                F.col(column_name),
                r",\s*"
            )
        )
    )


slv_user = (
    brz_user

    .select(

        F.col("user_id"),

        F.trim(
            F.col("name")
        ).alias("nombre"),

        F.to_timestamp(
            F.col("yelping_since")
        ).alias("yelping_since"),

        F.col("review_count")
            .cast("long")
            .alias("review_count"),

        F.col("useful")
            .cast("long")
            .alias("useful"),

        F.col("funny")
            .cast("long")
            .alias("funny"),

        F.col("cool")
            .cast("long")
            .alias("cool"),

        F.col("fans")
            .cast("long")
            .alias("fans"),

        F.col("average_stars")
            .cast("double")
            .alias("average_stars"),

        csv_to_array(
            "elite"
        ).alias("elite"),

        csv_to_array(
            "friends"
        ).alias("friends"),

        F.col("_archivo_origen"),

        F.col("_execution_id")
            .alias("_bronze_execution_id"),

        F.col("_record_checksum")
            .alias("_bronze_record_checksum"),

        F.lit(
            silver_execution_id
        ).alias("_silver_execution_id"),

        F.current_timestamp()
            .alias("_silver_processed_at"),

        F.lit("silver")
            .alias("_layer")
    )

    .filter(
        F.col("user_id")
        .isNotNull()
    )

    .dropDuplicates(
        ["user_id"]
    )
)


(
    slv_user

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_user"
    )
)


user_count = (
    spark.table(
        f"{SILVER}.slv_user"
    )
    .count()
)


print(
    f"✔ slv_user: "
    f"{user_count:,} filas"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6. REVIEW
# MAGIC
# MAGIC Esta es la entidad principal del caso Trust & Safety.
# MAGIC
# MAGIC Se realiza:
# MAGIC
# MAGIC - casteo de stars;
# MAGIC - date → timestamp;
# MAGIC - normalización del texto;
# MAGIC - validación de Business;
# MAGIC - detección de cobertura User;
# MAGIC - detección de duplicado por `review_id`;
# MAGIC - detección de duplicado lógico;
# MAGIC - separación entre Review válida y cuarentena.
# MAGIC
# MAGIC Importante:
# MAGIC
# MAGIC Un posible duplicado lógico **NO se elimina automáticamente**.
# MAGIC Puede convertirse posteriormente en una señal Trust & Safety.


# COMMAND ----------

# ============================================================
# 6. SILVER REVIEW - FUENTE
# ============================================================

brz_review = (
    spark.table(
        f"{BRONZE}.brz_review"
    )
)


# Business válidos Silver
business_keys = (
    spark.table(
        f"{SILVER}.slv_business"
    )

    .select(
        "business_id"
    )

    .dropDuplicates()

    .withColumn(
        "_business_existe",
        F.lit(True)
    )
)


# Users disponibles en la muestra
user_keys = (
    spark.table(
        f"{SILVER}.slv_user"
    )

    .select(
        "user_id"
    )

    .dropDuplicates()

    .withColumn(
        "_user_en_muestra",
        F.lit(True)
    )
)


# COMMAND ----------

# ============================================================
# 6.1 CONFORMACIÓN REVIEW
# ============================================================

review_base = (
    brz_review

    .select(

        F.col("review_id"),

        F.col("user_id"),

        F.col("business_id"),

        F.col("stars")
            .cast("double")
            .alias("stars"),

        F.trim(
            F.col("text")
        ).alias("review_text"),

        F.to_timestamp(
            F.col("date")
        ).alias("review_ts"),

        F.col("useful")
            .cast("long")
            .alias("useful"),

        F.col("funny")
            .cast("long")
            .alias("funny"),

        F.col("cool")
            .cast("long")
            .alias("cool"),

        F.col("_archivo_origen"),

        F.col("_execution_id")
            .alias("_bronze_execution_id"),

        F.col("_record_checksum")
            .alias("_bronze_record_checksum"),

        F.lit(
            silver_execution_id
        ).alias("_silver_execution_id"),

        F.current_timestamp()
            .alias("_silver_processed_at"),

        F.lit("silver")
            .alias("_layer")
    )
)


# COMMAND ----------

# ============================================================
# 6.2 INTEGRIDAD REFERENCIAL
# ============================================================

review_integridad = (
    review_base

    .join(
        business_keys,
        "business_id",
        "left"
    )

    .join(
        user_keys,
        "user_id",
        "left"
    )

    .fillna(
        {
            "_business_existe": False,
            "_user_en_muestra": False
        }
    )
)


# COMMAND ----------

# ============================================================
# 6.3 DUPLICADO POR REVIEW_ID
# ============================================================

w_review_id = (
    Window
    .partitionBy(
        "review_id"
    )
    .orderBy(
        F.col(
            "_bronze_record_checksum"
        ),
        F.col(
            "_archivo_origen"
        )
    )
)


review_dedup = (
    review_integridad

    .withColumn(
        "_review_id_row_number",

        F.row_number()
        .over(
            w_review_id
        )
    )

    .withColumn(
        "_duplicate_review_id",

        F.col(
            "_review_id_row_number"
        ) > 1
    )
)


# COMMAND ----------

# ============================================================
# 6.4 DUPLICADO LÓGICO
# ============================================================

review_dedup = (
    review_dedup

    .withColumn(
        "_logical_review_hash",

        F.sha2(

            F.concat_ws(

                "||",

                F.coalesce(
                    F.col("user_id"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("business_id"),
                    F.lit("")
                ),

                F.lower(
                    F.trim(
                        F.coalesce(
                            F.col("review_text"),
                            F.lit("")
                        )
                    )
                )
            ),

            256
        )
    )
)


w_logical = (
    Window
    .partitionBy(
        "_logical_review_hash"
    )
)


review_dedup = (
    review_dedup

    .withColumn(
        "_logical_duplicate_count",

        F.count("*")
        .over(
            w_logical
        )
    )

    .withColumn(
        "_possible_logical_duplicate",

        F.col(
            "_logical_duplicate_count"
        ) > 1
    )
)


# COMMAND ----------

# ============================================================
# 6.5 CLASIFICACIÓN PARA CUARENTENA
# ============================================================

review_clasificada = (
    review_dedup

    .withColumn(

        "_motivo_cuarentena",

        F.when(
            F.col("review_id").isNull(),

            F.lit(
                "REVIEW_ID_NULL"
            )
        )

        .when(
            F.col("business_id").isNull(),

            F.lit(
                "BUSINESS_ID_NULL"
            )
        )

        .when(
            ~F.col(
                "_business_existe"
            ),

            F.lit(
                "BUSINESS_NOT_FOUND"
            )
        )

        .when(
            F.col("stars").isNull(),

            F.lit(
                "STARS_NULL_OR_INVALID"
            )
        )

        .when(
            ~F.col(
                "stars"
            ).between(
                1.0,
                5.0
            ),

            F.lit(
                "STARS_OUT_OF_RANGE"
            )
        )

        .when(
            F.col(
                "review_ts"
            ).isNull(),

            F.lit(
                "INVALID_REVIEW_DATE"
            )
        )

        .when(
            F.col(
                "_duplicate_review_id"
            ),

            F.lit(
                "DUPLICATE_REVIEW_ID"
            )
        )

        .otherwise(
            F.lit(None)
        )
    )
)


# COMMAND ----------

# ============================================================
# 6.6 REVIEW VÁLIDA
# ============================================================

slv_review = (
    review_clasificada

    .filter(
        F.col(
            "_motivo_cuarentena"
        ).isNull()
    )

    .drop(
        "_review_id_row_number",
        "_motivo_cuarentena",
        "_duplicate_review_id"
    )
)


(
    slv_review

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_review"
    )
)


# COMMAND ----------

# ============================================================
# 6.7 REVIEW CUARENTENA
# ============================================================

slv_review_cuarentena = (
    review_clasificada

    .filter(
        F.col(
            "_motivo_cuarentena"
        ).isNotNull()
    )
)


(
    slv_review_cuarentena

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_review_cuarentena"
    )
)


review_count = (
    spark.table(
        f"{SILVER}.slv_review"
    )
    .count()
)


quarantine_count = (
    spark.table(
        f"{SILVER}.slv_review_cuarentena"
    )
    .count()
)


print(
    f"✔ slv_review: "
    f"{review_count:,} filas"
)


print(
    f"⚠ slv_review_cuarentena: "
    f"{quarantine_count:,} filas"
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Importante sobre User
# MAGIC
# MAGIC Si una Review contiene un `user_id` que no aparece en
# MAGIC `slv_user`, **no se envía automáticamente a cuarentena**.
# MAGIC
# MAGIC La muestra User puede no contener la totalidad de usuarios.
# MAGIC
# MAGIC El registro se conserva y se marca mediante:
# MAGIC
# MAGIC `_user_en_muestra = false`
# MAGIC
# MAGIC Este hecho será medido mediante una regla DQ de integridad.


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. TIP
# MAGIC
# MAGIC Yelp TIP no posee un `tip_id` nativo.
# MAGIC
# MAGIC Se genera un identificador determinístico SHA-256 utilizando:
# MAGIC
# MAGIC `user_id + business_id + date + text`.


# COMMAND ----------

# ============================================================
# 7. SILVER TIP
# ============================================================

brz_tip = (
    spark.table(
        f"{BRONZE}.brz_tip"
    )
)


slv_tip = (
    brz_tip

    .withColumn(
        "tip_id",

        F.sha2(

            F.concat_ws(
                "||",

                F.coalesce(
                    F.col("user_id"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("business_id"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("date"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("text"),
                    F.lit("")
                )
            ),

            256
        )
    )

    .select(

        F.col("tip_id"),

        F.col("user_id"),

        F.col("business_id"),

        F.trim(
            F.col("text")
        ).alias("tip_text"),

        F.to_timestamp(
            F.col("date")
        ).alias("tip_ts"),

        F.col("compliment_count")
            .cast("long")
            .alias("compliment_count"),

        F.col("_archivo_origen"),

        F.col("_execution_id")
            .alias("_bronze_execution_id"),

        F.col("_record_checksum")
            .alias("_bronze_record_checksum"),

        F.lit(
            silver_execution_id
        ).alias("_silver_execution_id"),

        F.current_timestamp()
            .alias("_silver_processed_at"),

        F.lit("silver")
            .alias("_layer")
    )

    .dropDuplicates(
        ["tip_id"]
    )
)


(
    slv_tip

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_tip"
    )
)


tip_count = (
    spark.table(
        f"{SILVER}.slv_tip"
    )
    .count()
)


print(
    f"✔ slv_tip: "
    f"{tip_count:,} filas"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. CHECK-IN
# MAGIC
# MAGIC En Bronze:
# MAGIC
# MAGIC `1 fila = Business + lista de fechas`
# MAGIC
# MAGIC En Silver:
# MAGIC
# MAGIC `1 fila = 1 evento Check-in`
# MAGIC
# MAGIC Por ello se utiliza:
# MAGIC
# MAGIC `split → explode → timestamp`


# COMMAND ----------

# ============================================================
# 8. SILVER CHECKIN EVENT
# ============================================================

brz_checkin = (
    spark.table(
        f"{BRONZE}.brz_checkin"
    )
)


slv_checkin_event = (
    brz_checkin

    .withColumn(
        "_checkin_raw",

        F.explode(
            F.split(
                F.col("date"),
                r",\s*"
            )
        )
    )

    .select(

        F.col("business_id"),

        F.to_timestamp(
            F.trim(
                F.col(
                    "_checkin_raw"
                )
            )
        ).alias(
            "checkin_ts"
        ),

        F.col("_archivo_origen"),

        F.col("_execution_id")
            .alias("_bronze_execution_id"),

        F.col("_record_checksum")
            .alias("_bronze_record_checksum"),

        F.lit(
            silver_execution_id
        ).alias("_silver_execution_id"),

        F.current_timestamp()
            .alias("_silver_processed_at"),

        F.lit("silver")
            .alias("_layer")
    )
)


(
    slv_checkin_event

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_checkin_event"
    )
)


checkin_count = (
    spark.table(
        f"{SILVER}.slv_checkin_event"
    )
    .count()
)


print(
    f"✔ slv_checkin_event: "
    f"{checkin_count:,} eventos"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Validación de resultados Silver


# COMMAND ----------

# ============================================================
# 9. RESUMEN SILVER
# ============================================================

resultados = [

    (
        "slv_business",
        business_count
    ),

    (
        "slv_user",
        user_count
    ),

    (
        "slv_review",
        review_count
    ),

    (
        "slv_review_cuarentena",
        quarantine_count
    ),

    (
        "slv_tip",
        tip_count
    ),

    (
        "slv_checkin_event",
        checkin_count
    )
]


df_resumen = (
    spark.createDataFrame(
        resultados,
        [
            "tabla",
            "filas"
        ]
    )
)


display(
    df_resumen
)


# COMMAND ----------

# ============================================================
# 10. MOTIVOS DE CUARENTENA REVIEW
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            _motivo_cuarentena,

            COUNT(*) AS total

        FROM
            {SILVER}.slv_review_cuarentena

        GROUP BY
            _motivo_cuarentena

        ORDER BY
            total DESC
        """
    )
)


# COMMAND ----------

# ============================================================
# 11. COBERTURA USER EN REVIEW
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            _user_en_muestra,

            COUNT(*) AS total,

            ROUND(
                COUNT(*) * 100.0
                /
                SUM(
                    COUNT(*)
                ) OVER(),
                2
            ) AS porcentaje

        FROM
            {SILVER}.slv_review

        GROUP BY
            _user_en_muestra
        """
    )
)


# COMMAND ----------

# ============================================================
# 12. POSIBLES DUPLICADOS LÓGICOS
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            _possible_logical_duplicate,

            COUNT(*) AS total

        FROM
            {SILVER}.slv_review

        GROUP BY
            _possible_logical_duplicate
        """
    )
)


# COMMAND ----------

# ============================================================
# 13. VALIDAR GRAIN CHECK-IN
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            COUNT(*) AS total_eventos,

            COUNT(
                DISTINCT business_id
            ) AS negocios,

            MIN(
                checkin_ts
            ) AS primer_checkin,

            MAX(
                checkin_ts
            ) AS ultimo_checkin

        FROM
            {SILVER}.slv_checkin_event
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — Silver
# MAGIC
# MAGIC - [ ] `slv_business` creada.
# MAGIC - [ ] `slv_user` creada.
# MAGIC - [ ] `slv_review` creada.
# MAGIC - [ ] `slv_review_cuarentena` creada.
# MAGIC - [ ] `slv_tip` creada.
# MAGIC - [ ] `slv_checkin_event` creada.
# MAGIC - [ ] Tipos de datos conformados.
# MAGIC - [ ] `review_id` deduplicado.
# MAGIC - [ ] Duplicados lógicos detectados sin eliminarlos.
# MAGIC - [ ] FK Business evaluada para Review.
# MAGIC - [ ] Cobertura User registrada.
# MAGIC - [ ] Check-ins llevados a grano evento.
# MAGIC - [ ] Linaje Bronze preservado.
# MAGIC
# MAGIC El siguiente paso es ejecutar:
# MAGIC
# MAGIC `E5_calidad_datos/03_dq_silver.py`


# COMMAND ----------

# ============================================================
# 14. RESULTADO FINAL
# ============================================================

execution_finished_at = (
    datetime
    .now(timezone.utc)
    .isoformat()
)


print()
print("=" * 70)
print("BRONZE -> SILVER FINALIZADO")
print("=" * 70)

print(
    f"Execution ID : "
    f"{silver_execution_id}"
)

print(
    f"Inicio UTC   : "
    f"{execution_started_at}"
)

print(
    f"Fin UTC      : "
    f"{execution_finished_at}"
)

print()

print(
    f"Business            : "
    f"{business_count:,}"
)

print(
    f"User                : "
    f"{user_count:,}"
)

print(
    f"Review válidas      : "
    f"{review_count:,}"
)

print(
    f"Review cuarentena   : "
    f"{quarantine_count:,}"
)

print(
    f"Tip                 : "
    f"{tip_count:,}"
)

print(
    f"Check-in eventos    : "
    f"{checkin_count:,}"
)

print("=" * 70)